# DL tuning

Ноутбук подбирает гиперпараметры DL-архитектур, выбранных после baseline-эксперимента.

Логика эксперимента:
- обучение и validation выполняются на типовых задачах (`duration_hours < 960`);
- основной критерий выбора конфигурации — `selected_val_MAE`;
- typical holdout и full holdout считаются после выбора модели;
- baseline-конфигурация сохраняется, если тюнинг не улучшает внутреннюю validation.


In [ ]:
import os

RUN_TABPFN_DEFAULT = os.environ.get("RUN_TABPFN", "1") not in {"0", "false", "False"}
TABPFN_ENABLE_HF_LOGIN = True
TABPFN_HF_TOKEN_RESOLVED = (
    os.environ.get("HF_TOKEN", "").strip()
    or os.environ.get("HUGGING_FACE_HUB_TOKEN", "").strip()
    or os.environ.get("HUGGINGFACEHUB_API_TOKEN", "").strip()
    or os.environ.get("HUGGINGFACE_TOKEN", "").strip()
)
TABPFN_HF_LOGIN_STATUS = "token_present" if TABPFN_HF_TOKEN_RESOLVED else "missing_token"


In [ ]:
from pathlib import Path
import json
import ast
import math
import gc
import random
import os
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

SEED = 2
DURATION_CAP = 960
TARGET_COL = "duration_hours"
DATA_PATH_ENV = "DATA_FINALL_PATH"

def seed_everything(seed: int = 2):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except Exception:
        pass

seed_everything(SEED)

data_path_value = os.environ.get(DATA_PATH_ENV)
if not data_path_value:
    raise RuntimeError(f"Перед запуском укажи путь к датасету в переменной окружения {DATA_PATH_ENV}.")

DATA_PATH = Path(data_path_value).expanduser()
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Файл из {DATA_PATH_ENV} не найден: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

if TARGET_COL not in df.columns:
    raise ValueError(f"В датасете нет целевой колонки {TARGET_COL!r}")

split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full = df.iloc[split:].copy()

train_typical = train_full[train_full[TARGET_COL] < DURATION_CAP].copy()
test_typical = test_full[test_full[TARGET_COL] < DURATION_CAP].copy()

Xtrain = train_typical.drop(columns=[TARGET_COL], errors="ignore").copy()
ytrain = train_typical[TARGET_COL].copy()

Xtest_typical = test_typical.drop(columns=[TARGET_COL], errors="ignore").copy()
ytest_typical = test_typical[TARGET_COL].copy()

Xtest_full = test_full.drop(columns=[TARGET_COL], errors="ignore").copy()
ytest_full = test_full[TARGET_COL].copy()

bool_cols = Xtrain.select_dtypes(include=["bool"]).columns.tolist()
for _frame in (Xtrain, Xtest_typical, Xtest_full):
    if bool_cols:
        _frame[bool_cols] = _frame[bool_cols].astype(np.uint8)

non_numeric = Xtrain.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    raise ValueError(
        "Ожидался уже подготовленный числовой датасет, но найдены нечисловые колонки: "
        + ", ".join(non_numeric[:10])
    )

Xtest = Xtest_typical.copy()
ytest = ytest_typical.copy()
seed = SEED

print(f"Всего строк:                 {len(df)}")
print(f"Train raw:                   {len(train_full)}")
print(f"Train typical < {DURATION_CAP}:        {len(train_typical)}")
print(f"Test typical < {DURATION_CAP}:         {len(test_typical)}")
print(f"Test full:                   {len(test_full)}")
print(f"Train outliers >= {DURATION_CAP}:      {(train_full[TARGET_COL] >= DURATION_CAP).sum()}")
print(f"Test outliers >= {DURATION_CAP}:       {(test_full[TARGET_COL] >= DURATION_CAP).sum()}")


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time, math, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from tabpfn import TabPFNRegressor
    HAS_TABPFN = True
    print("TabPFN доступен ✓")
    print(f"TabPFN HF login status: {globals().get('TABPFN_HF_LOGIN_STATUS', 'unknown')}")
except ImportError:
    HAS_TABPFN = False
    print("TabPFN не установлен; связанные блоки будут пропущены.")

torch.manual_seed(seed)
np.random.seed(seed)

#  1. Подготовка данных

val_cut = int(len(Xtrain) * 0.8)

X_dl_tr = Xtrain.iloc[:val_cut].values.astype(np.float32)
y_dl_tr = ytrain.iloc[:val_cut].values.astype(np.float32)
X_dl_va = Xtrain.iloc[val_cut:].values.astype(np.float32)
y_dl_va = ytrain.iloc[val_cut:].values.astype(np.float32)

sc = StandardScaler()
X_dl_tr = sc.fit_transform(X_dl_tr)
X_dl_va = sc.transform(X_dl_va)
X_dl_te = sc.transform(Xtest.values.astype(np.float32))

sc_full = StandardScaler()
X_full_s = sc_full.fit_transform(Xtrain.values.astype(np.float32))
X_te_full_s = sc_full.transform(Xtest.values.astype(np.float32))

for arr in [X_dl_tr, X_dl_va, X_dl_te, X_full_s, X_te_full_s]:
    np.nan_to_num(arr, copy=False)

X_tr_t = torch.from_numpy(X_dl_tr)
y_tr_t = torch.from_numpy(y_dl_tr).unsqueeze(1)
X_va_t = torch.from_numpy(X_dl_va)
y_va_t = torch.from_numpy(y_dl_va).unsqueeze(1)
X_te_t = torch.from_numpy(X_dl_te)
X_full_t = torch.from_numpy(X_full_s)
y_full_t = torch.from_numpy(ytrain.values.astype(np.float32)).unsqueeze(1)
X_te_full_t = torch.from_numpy(X_te_full_s)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)
INPUT_DIM = X_dl_tr.shape[1]
print(f"Device: {device},  Input dim: {INPUT_DIM}")
print(f"Train: {X_dl_tr.shape}, Val: {X_dl_va.shape}, Test: {X_dl_te.shape}\n")


#  2. АРХИТЕКТУРЫ  (24 шт.)

# 2.1  MLP

class MLP(nn.Module):
    def __init__(self, d_in, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


# 2.2  ResMLP

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.block(x))

class ResMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU())
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(self.proj(x)))


# 2.3  SNN

class SNN(nn.Module):
    def __init__(self, d_in, hidden_dims=[256, 128], alpha_dropout=0.1):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.SELU(), nn.AlphaDropout(alpha_dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)


# 2.4  GatedMLP

class GatedBlock(nn.Module):
    def __init__(self, d_in, d_out, dropout):
        super().__init__()
        self.linear = nn.Linear(d_in, d_out * 2)
        self.bn = nn.BatchNorm1d(d_out)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.linear(x)
        h1, h2 = h.chunk(2, dim=-1)
        return self.drop(self.bn(h1 * torch.sigmoid(h2)))

class GatedMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        blocks = [GatedBlock(d_in, hidden, dropout)]
        for _ in range(n_blocks - 1):
            blocks.append(GatedBlock(hidden, hidden, dropout))
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(x))


# 2.5  GANDALF

class GANDALF(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.gate_fc = nn.Linear(d_in, d_in)
        layers = [nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        for _ in range(n_blocks - 1):
            layers += [nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        self.dnn = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        gate = torch.sigmoid(self.gate_fc(x))
        return self.head(self.dnn(x * gate))


# 2.6  DAE-MLP

class DAEMLP(nn.Module):
    def __init__(self, d_in, hidden=256, latent=64, noise=0.3, dropout=0.3):
        super().__init__()
        self.noise = noise
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, latent), nn.BatchNorm1d(latent), nn.SiLU(),
        )
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.SiLU(), nn.Linear(hidden, d_in))
        self.reg_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(latent, 32), nn.SiLU(), nn.Linear(32, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x_in = x * (torch.rand_like(x) > self.noise).float() if self.training and self.noise > 0 else x
        z = self.encoder(x_in)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(z), x)
        return self.reg_head(z)


# 2.7  CNN1D

class CNN1D(nn.Module):
    def __init__(self, d_in, channels=[64, 128, 64], ks=5, dropout=0.3):
        super().__init__()
        layers = []
        in_ch = 1
        for ch in channels:
            layers += [nn.Conv1d(in_ch, ch, ks, padding=ks // 2),
                       nn.BatchNorm1d(ch), nn.SiLU(), nn.Dropout(dropout)]
            in_ch = ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Linear(channels[-1], 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.conv(x))
        return self.head(x.squeeze(-1))


# 2.8  BiGRU

class BiGRU(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden, num_layers=n_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        _, h = self.gru(x.unsqueeze(-1))
        return self.head(torch.cat([h[-2], h[-1]], dim=1))


# 2.9  FT-Transformer

class FTTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# 2.10  TabTransformer

class TabTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.col_emb = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_in * d_model + d_in, mlp_hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1)) + self.col_emb
        out = self.transformer(tok)
        pooled = out.reshape(out.size(0), -1)
        return self.head(torch.cat([pooled, x], dim=1))


# 2.11  SAINT

class SAINTBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.sa_norm = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ia_norm = nn.LayerNorm(d_model)
        self.inter_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model * 4),
                                 nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
    def forward(self, x):
        h = self.sa_norm(x)
        h, _ = self.self_attn(h, h, h)
        x = x + h
        B, F_, D = x.shape
        if B > 1 and self.training:
            xt = x.permute(1, 0, 2)
            h2 = self.ia_norm(xt)
            h2, _ = self.inter_attn(h2, h2, h2)
            x = (xt + h2).permute(1, 0, 2)
        return x + self.ffn(x)

class SAINT(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        self.blocks = nn.ModuleList([SAINTBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        for blk in self.blocks:
            tok = blk(tok)
        return self.head(tok[:, 0])


# 2.12  AutoInt

class AutoIntLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.W_res = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        h, _ = self.attn(x, x, x)
        return F.relu(self.norm(h + self.W_res(x)))

class AutoInt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([AutoIntLayer(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in * d_model, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        for layer in self.layers:
            tok = layer(tok)
        return self.head(tok.reshape(tok.size(0), -1))


# 2.13  Trompt

class TromptLayer(nn.Module):
    def __init__(self, d_in, d_model, n_heads, dropout):
        super().__init__()
        self.prompt = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.gate = nn.Linear(d_model, 1)
    def forward(self, feat_emb, x_raw):
        B = feat_emb.shape[0]
        prompted = feat_emb + self.prompt.expand(B, -1, -1)
        h, _ = self.attn(prompted, prompted, prompted)
        h = self.norm1(prompted + h)
        h = self.norm2(h + self.ffn(h))
        weights = torch.softmax(self.gate(h).squeeze(-1), dim=1)
        return h, weights * x_raw

class Trompt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([TromptLayer(d_in, d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        preds = []
        for layer in self.layers:
            tok, lp = layer(tok, x)
            preds.append(lp)
        return self.head(torch.stack(preds).mean(0))


# 2.14  ExcelFormer

class ExcelFormer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.importance = nn.Parameter(torch.zeros(d_in))
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        mask = torch.sigmoid(self.importance)
        x_masked = x * mask
        tok = self.feat_emb(x_masked.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# 2.15  ARM-Net

class ARMNet(nn.Module):
    def __init__(self, d_in, n_exp=64, hidden=128, order=2, dropout=0.2):
        super().__init__()
        self.order = order
        self.exp_layers = nn.ModuleList([nn.Linear(d_in, n_exp) for _ in range(order)])
        self.gate = nn.Sequential(nn.Linear(d_in, n_exp * order), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.BatchNorm1d(n_exp * order), nn.Linear(n_exp * order, hidden),
            nn.SiLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x):
        interactions = []
        for i, layer in enumerate(self.exp_layers):
            h = layer(x)
            if i > 0:
                h = h * interactions[-1]
            interactions.append(F.softplus(h))
        cat = torch.cat(interactions, dim=1)
        return self.head(cat * self.gate(x))


# 2.16  NODE

class NODE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_weights = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.01)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        choices = torch.sigmoid(torch.einsum('bj,tdj->btd', x, self.feat_weights) - self.thresholds)
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        return self.drop(tree_out).mean(dim=-1, keepdim=True)


# 2.17  GRANDE

class GRANDE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_logits = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.tree_weights = nn.Parameter(torch.ones(n_trees) / n_trees)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        feat_sel = F.softmax(self.feat_logits, dim=-1)
        proj = torch.einsum('bj,tdj->btd', x, feat_sel)
        choices = torch.sigmoid(10.0 * (proj - self.thresholds))
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        w = F.softmax(self.tree_weights, dim=0)
        return self.drop((tree_out * w).sum(dim=-1, keepdim=True))


# 2.18  Net-DNF

class NetDNF(nn.Module):
    def __init__(self, d_in, n_formulas=64, n_conjuncts=6, dropout=0.2):
        super().__init__()
        self.feat_sel = nn.Parameter(torch.randn(n_formulas, n_conjuncts, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_formulas, n_conjuncts))
        self.formula_w = nn.Linear(n_formulas, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        sel = torch.sigmoid(self.feat_sel)
        proj = torch.einsum('bj,fcj->bfc', x, sel)
        conjuncts = torch.sigmoid(proj - self.thresholds)
        formulas = conjuncts.prod(dim=-1)
        return self.formula_w(self.drop(formulas))


# 2.19  TabNet (simplified)

class TabNet(nn.Module):
    def __init__(self, d_in, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1):
        super().__init__()
        self.n_steps = n_steps
        self.relax = relax
        self.bn_init = nn.BatchNorm1d(d_in)
        self.shared_fc = nn.Linear(d_in, n_d + n_a)
        self.step_fcs = nn.ModuleList([nn.Linear(d_in, n_d + n_a) for _ in range(n_steps)])
        self.attn_fcs = nn.ModuleList([nn.Sequential(nn.Linear(n_a, d_in), nn.BatchNorm1d(d_in)) for _ in range(n_steps)])
        self.head = nn.Linear(n_d, 1)
        self.n_d = n_d
        self.drop = nn.Dropout(dropout)
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x = self.bn_init(x)
        prior_scales = torch.ones(x.shape[0], x.shape[1], device=x.device)
        agg = torch.zeros(x.shape[0], self.n_d, device=x.device)
        entropy_loss = 0.0
        for i in range(self.n_steps):
            h = self.shared_fc(x * prior_scales) + self.step_fcs[i](x * prior_scales)
            h_d, h_a = h[:, :self.n_d], h[:, self.n_d:]
            h_d = F.silu(h_d)
            agg = agg + self.drop(h_d)
            attn = self.attn_fcs[i](h_a)
            attn = F.softmax(attn * prior_scales, dim=-1)
            prior_scales = prior_scales * (self.relax - attn)
            entropy_loss += (-attn * torch.log(attn + 1e-15)).mean()
        if self.training:
            self.aux_loss = entropy_loss / self.n_steps
        return self.head(agg)


# 2.20  TabR

class TabR(nn.Module):
    def __init__(self, d_in, hidden=256, n_layers=3, k=32, dropout=0.3):
        super().__init__()
        self.k = k
        layers = []
        prev = d_in
        for i in range(n_layers):
            h = hidden if i == 0 else hidden // 2
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        self.encoder = nn.Sequential(*layers)
        self.latent_dim = prev
        self.retrieval_head = nn.Sequential(nn.Linear(prev * 2, 128), nn.SiLU(), nn.Dropout(dropout), nn.Linear(128, 1))
        self.direct_head = nn.Linear(prev, 1)
        self._store_z = None
        self._store_y = None
    def build_store(self, x_train, y_train):
        self.eval()
        with torch.no_grad():
            self._store_z = self.encoder(x_train)
            self._store_y = y_train
    def forward(self, x):
        z = self.encoder(x)
        if self._store_z is not None and not self.training:
            dists = torch.cdist(z, self._store_z)
            _, idx = dists.topk(self.k, largest=False)
            nbr_z = self._store_z[idx]
            sim = -dists.gather(1, idx)
            attn = torch.softmax(sim, dim=1).unsqueeze(-1)
            context = (attn * nbr_z).sum(dim=1)
            return self.retrieval_head(torch.cat([z, context], dim=1))
        return self.direct_head(z)


# 2.21  GrowNet stages

class GrowNetStage(nn.Module):
    def __init__(self, d_in, hidden=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in + 1, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 1))
    def forward(self, x, prev_pred):
        return self.net(torch.cat([x, prev_pred], dim=1))


# 2.22  SwitchTab

class SwitchTab(nn.Module):
    def __init__(self, d_in, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_in, d_enc), nn.BatchNorm1d(d_enc), nn.SiLU(), nn.Dropout(dropout))
        self.mutual_proj = nn.Linear(d_enc, d_mutual)
        self.salient_proj = nn.Linear(d_enc, d_salient)
        self.decoder = nn.Sequential(nn.Linear(d_mutual + d_salient, d_in))
        self.head = nn.Sequential(nn.Linear(d_mutual + d_salient, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        h = self.encoder(x)
        mutual = self.mutual_proj(h)
        salient = self.salient_proj(h)
        rep = torch.cat([mutual, salient], dim=1)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(rep), x)
        return self.head(rep)


# 2.23  PTaRL

class PTaRL(nn.Module):
    def __init__(self, d_in, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_latent), nn.BatchNorm1d(d_latent), nn.SiLU())
        self.prototypes = nn.Parameter(torch.randn(n_prototypes, d_latent) * 0.1)
        self.head = nn.Sequential(nn.Linear(d_latent * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        z = self.encoder(x)
        sim = F.cosine_similarity(z.unsqueeze(1), self.prototypes.unsqueeze(0), dim=-1)
        attn = F.softmax(sim * 10, dim=1)
        proto_rep = (attn.unsqueeze(-1) * self.prototypes.unsqueeze(0)).sum(dim=1)
        return self.head(torch.cat([z, proto_rep], dim=1))


# 2.24  DCN v2

class CrossLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.W = nn.Linear(d, d, bias=False)
        self.b = nn.Parameter(torch.zeros(d))
    def forward(self, x0, x):
        return x0 * self.W(x) + self.b + x

class DCNv2(nn.Module):
    def __init__(self, d_in, n_cross=3, hidden=256, dropout=0.3):
        super().__init__()
        self.cross_layers = nn.ModuleList([CrossLayer(d_in) for _ in range(n_cross)])
        self.deep = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.SiLU(), nn.Dropout(dropout))
        self.head = nn.Linear(d_in + hidden // 2, 1)
    def forward(self, x):
        x_cross = x
        for cl in self.cross_layers:
            x_cross = cl(x, x_cross)
        return self.head(torch.cat([x_cross, self.deep(x)], dim=1))


#  3. ФУНКЦИИ ОБУЧЕНИЯ

def train_model(model, epochs=300, lr=1e-3, wd=1e-4, bs=256,
                patience=30, aux_w=0.0, trial=None):
    """Обучение с early stopping. Возвращает (model, val_mae, epochs_used)."""
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    X_v, y_v = X_va_t.to(device), y_va_t.to(device)

    best_val, best_state, wait = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v_mae = loss_fn(model(X_v), y_v).item()
        sched.step(v_mae)
        if v_mae < best_val:
            best_val = v_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
        if trial is not None:
            trial.report(v_mae, ep)
            if trial.should_prune():
                raise optuna.TrialPruned()
    model.load_state_dict(best_state)
    return model, best_val, ep


def train_grownet(n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                  step_size=0.1, bs=256, patience=30, dropout=0.1, trial=None):
    """Gradient boosting с NN base learners. Возвращает (stages, val_mae, total_epochs)."""
    stages = []
    X_v, y_v = X_va_t.to(device), y_va_t.to(device)
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    # Предсказание ансамбля на train/val
    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    best_val, best_n = float("inf"), 0
    total_ep = 0
    for stage_idx in range(n_stages):
        model = GrowNetStage(INPUT_DIM, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
        best_stage_val, best_stage_state, wait = float("inf"), None, 0
        for ep in range(1, 201):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad(); loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                prev_v = ensemble_pred(X_v, stages)
                v_pred = prev_v + step_size * model(X_v, prev_v)
                v_mae = loss_fn(v_pred, y_v).item()
            sched.step(v_mae)
            if v_mae < best_stage_val:
                best_stage_val = v_mae
                best_stage_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break
            total_ep += 1
        model.load_state_dict(best_stage_state)
        stages.append(model)
        # Проверяем ансамбль
        with torch.no_grad():
            ens_val = loss_fn(ensemble_pred(X_v, stages), y_v).item()
        if ens_val < best_val:
            best_val = ens_val
            best_n = len(stages)
        if trial is not None:
            trial.report(ens_val, stage_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
    stages = stages[:best_n]
    return stages, best_val, total_ep


def eval_on_test(model, name=""):
    """Evaluate model on holdout test."""
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t.to(device)).cpu().numpy().flatten()
    mae = mean_absolute_error(ytest, pred)
    rmse = np.sqrt(mean_squared_error(ytest, pred))
    mdae = median_absolute_error(ytest, pred)
    return mae, rmse, mdae


def eval_grownet_on_test(stages, step_size=0.1):
    """Evaluate GrowNet ensemble on holdout test."""
    X_t = X_te_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    mae = mean_absolute_error(ytest, pred)
    rmse = np.sqrt(mean_squared_error(ytest, pred))
    mdae = median_absolute_error(ytest, pred)
    return mae, rmse, mdae


print("24 архитектуры + train_model + train_grownet определены.")


In [ ]:
# Дополнительные тензоры/функции для полного holdout (дополнительная проверка),
# чтобы не ломать исходные train/eval-хелперы из исходного ноутбука.

X_dl_te_full = sc.transform(Xtest_full.values.astype(np.float32))
np.nan_to_num(X_dl_te_full, copy=False)
X_te_stress_t = torch.from_numpy(X_dl_te_full)

X_te_fullscale_stress = sc_full.transform(Xtest_full.values.astype(np.float32))
np.nan_to_num(X_te_fullscale_stress, copy=False)
X_te_fullscale_stress_t = torch.from_numpy(X_te_fullscale_stress)

def clear_device_cache():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()

def calc_reg_metrics(y_true, pred):
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MdAE": float(median_absolute_error(y_true, pred)),
    }

def eval_on_typical_holdout(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_on_full_holdout(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_stress_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def eval_grownet_on_typical_holdout(stages, step_size=0.1):
    X_t = X_te_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_grownet_on_full_holdout(stages, step_size=0.1):
    X_t = X_te_stress_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def eval_tabpfn_holdouts(tabpfn_model, use_full_train_scaler=False):
    if use_full_train_scaler:
        X_typ = X_te_full_t.numpy()
        X_full = X_te_fullscale_stress_t.numpy()
    else:
        X_typ = X_te_t.numpy()
        X_full = X_te_stress_t.numpy()
    pred_typ = tabpfn_model.predict(X_typ)
    pred_full = tabpfn_model.predict(X_full)
    return calc_reg_metrics(ytest_typical, pred_typ), calc_reg_metrics(ytest_full, pred_full)

def train_model_fulltrain(model, epochs=100, lr=1e-3, wd=1e-4, bs=256, aux_w=0.0):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_full_t, y_full_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
    return model

def eval_fulltrain_model_typical(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_full_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_fulltrain_model_full(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_fullscale_stress_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def train_grownet_fulltrain(n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                            step_size=0.1, bs=256, epochs_per_stage=50, dropout=0.1):
    stages = []
    ds = TensorDataset(X_full_t, y_full_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    for stage_idx in range(n_stages):
        model = GrowNetStage(INPUT_DIM, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs_per_stage, 1), eta_min=1e-6)

        for ep in range(epochs_per_stage):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad()
                loss.backward()
                opt.step()
            sched.step()

        stages.append(model)
    return stages

def eval_fulltrain_grownet_typical(stages, step_size=0.1):
    X_t = X_te_full_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_fulltrain_grownet_full(stages, step_size=0.1):
    X_t = X_te_fullscale_stress_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

print("Функции для полного holdout определены.")


In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

#  Baseline: все 24 архитектуры с дефолтными параметрами

DEFAULTS = {
    # MLP-семейство
    "MLP":            lambda: MLP(INPUT_DIM, [256, 128], dropout=0.3),
    "ResMLP":         lambda: ResMLP(INPUT_DIM, hidden=256, n_blocks=3, dropout=0.3),
    "SNN":            lambda: SNN(INPUT_DIM, hidden_dims=[256, 128], alpha_dropout=0.1),
    "GatedMLP":       lambda: GatedMLP(INPUT_DIM, hidden=256, n_blocks=3, dropout=0.3),
    "GANDALF":        lambda: GANDALF(INPUT_DIM, hidden=256, n_blocks=3, dropout=0.3),
    "DAE-MLP":        lambda: DAEMLP(INPUT_DIM, hidden=256, latent=64, noise=0.3, dropout=0.3),
    # CNN / RNN
    "CNN1D":          lambda: CNN1D(INPUT_DIM, channels=[64, 128, 64], ks=5, dropout=0.3),
    "BiGRU":          lambda: BiGRU(INPUT_DIM, hidden=64, n_layers=2, dropout=0.3),
    # Transformer / Attention
    "FT-Transformer": lambda: FTTransformer(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
    "TabTransformer": lambda: TabTransformer(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2),
    "SAINT":          lambda: SAINT(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
    "AutoInt":        lambda: AutoInt(INPUT_DIM, d_model=32, n_heads=4, n_layers=3, dropout=0.2),
    "Trompt":         lambda: Trompt(INPUT_DIM, d_model=32, n_heads=4, n_layers=3, dropout=0.2),
    "ExcelFormer":    lambda: ExcelFormer(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
    "ARM-Net":        lambda: ARMNet(INPUT_DIM, n_exp=64, hidden=128, order=2, dropout=0.2),
    # Tree-inspired
    "NODE":           lambda: NODE(INPUT_DIM, n_trees=32, depth=4, dropout=0.0),
    "GRANDE":         lambda: GRANDE(INPUT_DIM, n_trees=32, depth=4, dropout=0.0),
    "Net-DNF":        lambda: NetDNF(INPUT_DIM, n_formulas=64, n_conjuncts=6, dropout=0.2),
    "TabNet":         lambda: TabNet(INPUT_DIM, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1),
    # Retrieval / Special
    "TabR":           lambda: TabR(INPUT_DIM, hidden=256, n_layers=3, k=32, dropout=0.3),
    "SwitchTab":      lambda: SwitchTab(INPUT_DIM, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3),
    "PTaRL":          lambda: PTaRL(INPUT_DIM, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3),
    "DCNv2":          lambda: DCNv2(INPUT_DIM, n_cross=3, hidden=256, dropout=0.3),
}

AUX_W = {"DAE-MLP": 0.1, "TabNet": 0.01, "SwitchTab": 0.1}


In [ ]:
def build_arch_from_params(arch_name: str, bp: dict):
    if arch_name == "MLP":
        dims = [max(int(bp["first_hidden"]) // (2 ** i), 32) for i in range(int(bp["n_layers"]))]
        return MLP(INPUT_DIM, dims, bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ResMLP":
        return ResMLP(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SNN":
        dims = [max(int(bp["first_hidden"]) // (2 ** i), 32) for i in range(int(bp["n_layers"]))]
        return SNN(INPUT_DIM, dims, bp["alpha_dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GatedMLP":
        return GatedMLP(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GANDALF":
        return GANDALF(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "DAE-MLP":
        return DAEMLP(INPUT_DIM, bp["hidden"], bp["latent"], bp["noise"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "CNN1D":
        nc = int(bp["n_conv"]); bc = int(bp["base_ch"])
        chs = [bc * (2 ** i) for i in range(nc // 2 + 1)]
        chs += [bc * (2 ** i) for i in range(nc // 2 - 1, -1, -1)]
        chs = chs[:nc]
        return CNN1D(INPUT_DIM, chs, bp["ks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "BiGRU":
        return BiGRU(INPUT_DIM, bp["hidden"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "FT-Transformer":
        return FTTransformer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabTransformer":
        return TabTransformer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["mlp_hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SAINT":
        return SAINT(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "AutoInt":
        return AutoInt(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "Trompt":
        return Trompt(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ExcelFormer":
        return ExcelFormer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ARM-Net":
        return ARMNet(INPUT_DIM, bp["n_exp"], bp["hidden"], bp["order"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "NODE":
        return NODE(INPUT_DIM, bp["n_trees"], bp["depth"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GRANDE":
        return GRANDE(INPUT_DIM, bp["n_trees"], bp["depth"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "Net-DNF":
        return NetDNF(INPUT_DIM, bp["n_formulas"], bp["n_conjuncts"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabNet":
        return TabNet(INPUT_DIM, bp["n_steps"], bp["n_d"], bp["n_a"], bp["relax"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabR":
        return TabR(INPUT_DIM, bp["hidden"], bp["n_layers"], bp["k"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SwitchTab":
        return SwitchTab(INPUT_DIM, bp["d_enc"], bp["d_mutual"], bp["d_salient"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "PTaRL":
        return PTaRL(INPUT_DIM, bp["n_prototypes"], bp["d_latent"], bp["hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "DCNv2":
        return DCNv2(INPUT_DIM, bp["n_cross"], bp["hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    else:
        raise KeyError(f"Unknown architecture: {arch_name}")

def serialize_params(params: dict):
    clean = {}
    for k, v in params.items():
        if isinstance(v, (np.integer,)):
            clean[k] = int(v)
        elif isinstance(v, (np.floating,)):
            clean[k] = float(v)
        else:
            clean[k] = v
    return clean


In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

#  Optuna objectives + baseline-aware honest tuning

BASELINE_TRIAL_PARAMS = {
    "MLP":            {"n_layers": 2, "first_hidden": 256, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "ResMLP":         {"hidden": 256, "n_blocks": 3, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "SNN":            {"n_layers": 2, "first_hidden": 256, "alpha_dropout": 0.1, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "GatedMLP":       {"hidden": 256, "n_blocks": 3, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "GANDALF":        {"hidden": 256, "n_blocks": 3, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "DAE-MLP":        {"hidden": 256, "latent": 64, "noise": 0.3, "dropout": 0.3, "aux_w": 0.1, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "CNN1D":          {"n_conv": 3, "base_ch": 64, "ks": 5, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "BiGRU":          {"hidden": 64, "n_layers": 2, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "FT-Transformer": {"d_model": 32, "n_heads": 4, "n_layers": 2, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "TabTransformer": {"d_model": 32, "n_heads": 4, "n_layers": 2, "mlp_hidden": 128, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "SAINT":          {"d_model": 32, "n_heads": 4, "n_layers": 2, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "AutoInt":        {"d_model": 32, "n_heads": 4, "n_layers": 3, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "Trompt":         {"d_model": 32, "n_heads": 4, "n_layers": 3, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "ExcelFormer":    {"d_model": 32, "n_heads": 4, "n_layers": 2, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "ARM-Net":        {"n_exp": 64, "hidden": 128, "order": 2, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "NODE":           {"n_trees": 32, "depth": 4, "dropout": 0.0, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "GRANDE":         {"n_trees": 32, "depth": 4, "dropout": 0.0, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "Net-DNF":        {"n_formulas": 64, "n_conjuncts": 6, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "TabNet":         {"n_steps": 3, "n_d": 64, "n_a": 64, "relax": 1.5, "dropout": 0.1, "aux_w": 0.01, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "TabR":           {"hidden": 256, "n_layers": 3, "k": 32, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "SwitchTab":      {"d_enc": 128, "d_mutual": 64, "d_salient": 64, "dropout": 0.3, "aux_w": 0.1, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "PTaRL":          {"n_prototypes": 16, "d_latent": 128, "hidden": 256, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "DCNv2":          {"n_cross": 3, "hidden": 256, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "GrowNet":        {"n_stages": 5, "hidden": 32, "step_size": 0.1, "dropout": 0.1, "lr": 0.01, "wd": 1e-4, "bs": 256},
    "TabPFN":         {"n_estimators": 8, "max_rows": 3000},
}

def _seed_trial():
    # Один и тот же seed для каждого trial -> сравнение гиперпараметров честнее и ближе к baseline.
    seed_everything(SEED)

def _suggest_opt(trial, *, bs_choices, lr_low=1e-4, lr_high=3e-3, wd_low=1e-5, wd_high=1e-3):
    lr = trial.suggest_float("lr", lr_low, lr_high, log=True)
    wd = trial.suggest_float("wd", wd_low, wd_high, log=True)
    bs = trial.suggest_categorical("bs", bs_choices)
    return lr, wd, bs

def obj_mlp(trial):
    _seed_trial()
    n_layers = trial.suggest_int("n_layers", 2, 4)
    first = trial.suggest_categorical("first_hidden", [128, 256, 512])
    dims = [max(first // (2**i), 32) for i in range(n_layers)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(MLP(INPUT_DIM, dims, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_resmlp(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    n_blocks = trial.suggest_int("n_blocks", 2, 5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(ResMLP(INPUT_DIM, hidden, n_blocks, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_snn(trial):
    _seed_trial()
    n_layers = trial.suggest_int("n_layers", 2, 4)
    first = trial.suggest_categorical("first_hidden", [128, 256, 512])
    dims = [max(first // (2**i), 32) for i in range(n_layers)]
    alpha_do = trial.suggest_float("alpha_dropout", 0.01, 0.2)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512], wd_low=1e-6)
    return train_model(SNN(INPUT_DIM, dims, alpha_do), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_gatedmlp(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    n_blocks = trial.suggest_int("n_blocks", 2, 5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(GatedMLP(INPUT_DIM, hidden, n_blocks, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_gandalf(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    n_blocks = trial.suggest_int("n_blocks", 2, 5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(GANDALF(INPUT_DIM, hidden, n_blocks, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_daemlp(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 512])
    latent = trial.suggest_categorical("latent", [32, 64, 128])
    noise = trial.suggest_float("noise", 0.1, 0.5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    aux_w = trial.suggest_float("aux_w", 0.01, 0.5, log=True)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(DAEMLP(INPUT_DIM, hidden, latent, noise, dropout), lr=lr, wd=wd, bs=bs, aux_w=aux_w, trial=trial)[1]

def obj_cnn1d(trial):
    _seed_trial()
    n_conv = trial.suggest_int("n_conv", 2, 4)
    base_ch = trial.suggest_categorical("base_ch", [32, 64, 128])
    chs = [base_ch * (2**i) for i in range(n_conv // 2 + 1)]
    chs += [base_ch * (2**i) for i in range(n_conv // 2 - 1, -1, -1)]
    chs = chs[:n_conv]
    ks = trial.suggest_categorical("ks", [3, 5, 7, 9])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(CNN1D(INPUT_DIM, chs, ks, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_bigru(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [32, 64, 128])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256], wd_low=1e-5)
    return train_model(BiGRU(INPUT_DIM, hidden, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_ft_transformer(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32, 48])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(FTTransformer(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_tabtransformer(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    mlp_hidden = trial.suggest_categorical("mlp_hidden", [64, 128, 256])
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(TabTransformer(INPUT_DIM, d_model, n_heads, n_layers, mlp_hidden, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_saint(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(SAINT(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_autoint(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [8, 16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(AutoInt(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_trompt(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 2, 4)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(Trompt(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_excelformer(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(ExcelFormer(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_armnet(trial):
    _seed_trial()
    n_exp = trial.suggest_categorical("n_exp", [32, 64, 128])
    hidden = trial.suggest_categorical("hidden", [64, 128, 256])
    order = trial.suggest_int("order", 2, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256])
    return train_model(ARMNet(INPUT_DIM, n_exp, hidden, order, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_node(trial):
    _seed_trial()
    n_trees = trial.suggest_categorical("n_trees", [16, 32, 64, 128])
    depth = trial.suggest_int("depth", 3, 6)
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512], wd_low=1e-6)
    return train_model(NODE(INPUT_DIM, n_trees, depth, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_grande(trial):
    _seed_trial()
    n_trees = trial.suggest_categorical("n_trees", [16, 32, 64])
    depth = trial.suggest_int("depth", 3, 5)
    dropout = trial.suggest_float("dropout", 0.0, 0.2)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512], wd_low=1e-6)
    return train_model(GRANDE(INPUT_DIM, n_trees, depth, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_netdnf(trial):
    _seed_trial()
    n_formulas = trial.suggest_categorical("n_formulas", [32, 64, 128])
    n_conjuncts = trial.suggest_int("n_conjuncts", 4, 8)
    dropout = trial.suggest_float("dropout", 0.1, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(NetDNF(INPUT_DIM, n_formulas, n_conjuncts, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_tabnet(trial):
    _seed_trial()
    n_steps = trial.suggest_int("n_steps", 2, 5)
    n_d = trial.suggest_categorical("n_d", [32, 64, 128])
    n_a = trial.suggest_categorical("n_a", [32, 64, 128])
    relax = trial.suggest_float("relax", 1.0, 2.0)
    dropout = trial.suggest_float("dropout", 0.01, 0.2)
    aux_w = trial.suggest_float("aux_w", 0.001, 0.1, log=True)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256])
    return train_model(TabNet(INPUT_DIM, n_steps, n_d, n_a, relax, dropout), lr=lr, wd=wd, bs=bs, aux_w=aux_w, trial=trial)[1]

def obj_tabr(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    n_layers = trial.suggest_int("n_layers", 2, 4)
    k = trial.suggest_categorical("k", [16, 32, 64])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256])
    return train_model(TabR(INPUT_DIM, hidden, n_layers, k, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_switchtab(trial):
    _seed_trial()
    d_enc = trial.suggest_categorical("d_enc", [64, 128, 256])
    d_mutual = trial.suggest_categorical("d_mutual", [32, 64, 128])
    d_salient = trial.suggest_categorical("d_salient", [32, 64, 128])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    aux_w = trial.suggest_float("aux_w", 0.01, 0.5, log=True)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(SwitchTab(INPUT_DIM, d_enc, d_mutual, d_salient, dropout), lr=lr, wd=wd, bs=bs, aux_w=aux_w, trial=trial)[1]

def obj_ptarl(trial):
    _seed_trial()
    n_proto = trial.suggest_categorical("n_prototypes", [8, 16, 32])
    d_latent = trial.suggest_categorical("d_latent", [64, 128, 256])
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(PTaRL(INPUT_DIM, n_proto, d_latent, hidden, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_dcnv2(trial):
    _seed_trial()
    n_cross = trial.suggest_int("n_cross", 2, 5)
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(DCNv2(INPUT_DIM, n_cross, hidden, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_grownet(trial):
    _seed_trial()
    n_stages = trial.suggest_int("n_stages", 3, 8)
    hidden = trial.suggest_categorical("hidden", [16, 32, 64])
    step_size = trial.suggest_float("step_size", 0.05, 0.3)
    dropout = trial.suggest_float("dropout", 0.0, 0.2)
    lr = trial.suggest_float("lr", 1e-3, 0.05, log=True)
    wd = trial.suggest_float("wd", 1e-5, 1e-3, log=True)
    bs = trial.suggest_categorical("bs", [128, 256, 512])
    _, val_mae, _ = train_grownet(n_stages, hidden, lr, wd, step_size, bs, patience=30, dropout=dropout, trial=trial)
    return val_mae

def obj_tabpfn(trial):
    """TabPFN: baseline-aware tuning on the same inner train/val split as baseline."""
    _seed_trial()
    n_est = trial.suggest_int("n_estimators", 2, 32)
    max_rows = trial.suggest_categorical("max_rows", [1000, 2000, 3000])
    pfn_device = "cuda" if torch.cuda.is_available() else "cpu"

    Xp = X_dl_tr.copy()
    yp = y_dl_tr.copy()
    if len(Xp) > max_rows:
        idx = np.random.RandomState(SEED).choice(len(Xp), max_rows, replace=False)
        Xp = Xp[idx]
        yp = yp[idx]
    np.nan_to_num(Xp, copy=False)

    Xv = X_dl_va.copy()
    yv = y_dl_va.copy()
    np.nan_to_num(Xv, copy=False)

    model = TabPFNRegressor(device=pfn_device, n_estimators=n_est)
    model.fit(Xp, yp)
    pred = model.predict(Xv)
    return mean_absolute_error(yv, pred)


In [ ]:
# Artifacts / save locations
ART_DIR = Path("./artifacts_dl_tuning_final")
ART_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = ART_DIR / f"run_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

RUN_TABPFN = RUN_TABPFN_DEFAULT and HAS_TABPFN

ENQUEUE_BASELINE_TRIAL = True
KEEP_BASELINE_IF_NOT_IMPROVED = True
BASELINE_KEEP_EPS = 1e-9
TOP_K = 8
TUNE_MODE = "all"
MANUAL_MODELS = []
FORCE_INCLUDE_TABPFN_IN_TUNING = True

TRIAL_SCALE = 1.0
MIN_TRIALS = 8

BASELINE_DIR_ENV = "DL_BASELINE_ARTIFACTS_DIR"
baseline_dir_value = os.environ.get(BASELINE_DIR_ENV)
BASELINE_DIR = Path(baseline_dir_value).expanduser() if baseline_dir_value else Path("./artifacts_dl_baseline_final")

baseline_df = None
baseline_file = BASELINE_DIR / "baseline_by_val.csv"
if baseline_file.exists():
    baseline_df = pd.read_csv(baseline_file)
else:
    print("Baseline summary is not available; tuning will use TUNE_MODE rules without a baseline table.")

all_core_archs = list(DEFAULTS.keys()) + ["GrowNet"]
if RUN_TABPFN and HAS_TABPFN:
    all_core_archs = all_core_archs + ["TabPFN"]

if TUNE_MODE == "manual":
    selected_archs = [m for m in MANUAL_MODELS if m in all_core_archs]
elif TUNE_MODE == "all" or baseline_df is None:
    selected_archs = all_core_archs
else:
    baseline_candidates = [m for m in baseline_df["model"].tolist() if m in all_core_archs]
    selected_archs = baseline_candidates[:min(TOP_K, len(baseline_candidates))]

if FORCE_INCLUDE_TABPFN_IN_TUNING and RUN_TABPFN and HAS_TABPFN and "TabPFN" not in selected_archs:
    selected_archs.append("TabPFN")

print("Selected architectures for tuning:")
print(selected_archs)

ARCH_TRIALS_DEFAULT = {
    # Light / fast
    "MLP": 40,
    "SNN": 40,
    "GatedMLP": 40,
    "GANDALF": 40,
    "DCNv2": 40,
    # Medium
    "ResMLP": 30,
    "CNN1D": 30,
    "BiGRU": 30,
    "DAE-MLP": 30,
    "TabNet": 30,
    "Net-DNF": 30,
    "ARM-Net": 30,
    "PTaRL": 30,
    "SwitchTab": 30,
    "TabR": 30,
    # Heavy / slow
    "FT-Transformer": 25,
    "TabTransformer": 25,
    "SAINT": 25,
    "AutoInt": 25,
    "Trompt": 25,
    "ExcelFormer": 25,
    "NODE": 25,
    "GRANDE": 25,
    # Extra
    "GrowNet": 20,
    "TabPFN": 20,
}
ARCH_TRIALS = {k: 50 for k, v in ARCH_TRIALS_DEFAULT.items()}


In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

OBJECTIVE_MAP = {
    "MLP": obj_mlp,
    "ResMLP": obj_resmlp,
    "SNN": obj_snn,
    "GatedMLP": obj_gatedmlp,
    "GANDALF": obj_gandalf,
    "DAE-MLP": obj_daemlp,
    "CNN1D": obj_cnn1d,
    "BiGRU": obj_bigru,
    "FT-Transformer": obj_ft_transformer,
    "TabTransformer": obj_tabtransformer,
    "SAINT": obj_saint,
    "AutoInt": obj_autoint,
    "Trompt": obj_trompt,
    "ExcelFormer": obj_excelformer,
    "ARM-Net": obj_armnet,
    "NODE": obj_node,
    "GRANDE": obj_grande,
    "Net-DNF": obj_netdnf,
    "TabNet": obj_tabnet,
    "TabR": obj_tabr,
    "SwitchTab": obj_switchtab,
    "PTaRL": obj_ptarl,
    "DCNv2": obj_dcnv2,
    "GrowNet": obj_grownet,
}
if RUN_TABPFN and HAS_TABPFN:
    OBJECTIVE_MAP["TabPFN"] = obj_tabpfn

def fit_best_inner_model(arch_name: str, bp: dict):
    seed_everything(SEED)
    if arch_name == "GrowNet":
        stages, val_mae, epochs_used = train_grownet(
            n_stages=bp["n_stages"],
            hidden=bp["hidden"],
            lr=bp["lr"],
            wd=bp["wd"],
            step_size=bp["step_size"],
            bs=bp["bs"],
            patience=30,
            dropout=bp["dropout"],
        )
        metrics_typ = eval_grownet_on_typical_holdout(stages, bp["step_size"])
        metrics_full = eval_grownet_on_full_holdout(stages, bp["step_size"])
        n_params = int(sum(sum(p.numel() for p in s.parameters()) for s in stages))
        return {
            "kind": "grownet",
            "fitted": stages,
            "val_mae": float(val_mae),
            "epochs_used": int(epochs_used),
            "n_params": n_params,
            "metrics_typical": metrics_typ,
            "metrics_full": metrics_full,
        }

    if arch_name == "TabPFN":
        pfn_device = "cuda" if torch.cuda.is_available() else "cpu"
        max_rows = bp.get("max_rows", 3000)
        n_est = bp.get("n_estimators", 8)
        X_pfn_tr = X_dl_tr.copy()
        y_pfn_tr = y_dl_tr.copy()
        if len(X_pfn_tr) > max_rows:
            idx = np.random.RandomState(SEED).choice(len(X_pfn_tr), max_rows, replace=False)
            X_pfn_tr = X_pfn_tr[idx]
            y_pfn_tr = y_pfn_tr[idx]
        tabpfn = TabPFNRegressor(device=pfn_device, n_estimators=n_est)
        tabpfn.fit(X_pfn_tr, y_pfn_tr)
        val_pred = tabpfn.predict(X_dl_va)
        val_mae = mean_absolute_error(y_dl_va, val_pred)
        metrics_typ, metrics_full = eval_tabpfn_holdouts(tabpfn_model=tabpfn, use_full_train_scaler=False)
        return {
            "kind": "tabpfn",
            "fitted": tabpfn,
            "val_mae": float(val_mae),
            "epochs_used": 0,
            "n_params": 0,
            "metrics_typical": metrics_typ,
            "metrics_full": metrics_full,
        }

    model, aux_w = build_arch_from_params(arch_name, bp)
    model, val_mae, epochs_used = train_model(
        model,
        epochs=300,
        lr=bp["lr"],
        wd=bp["wd"],
        bs=bp["bs"],
        patience=30,
        aux_w=aux_w,
    )
    if arch_name == "TabR":
        model.build_store(X_tr_t.to(device), y_tr_t.to(device))
    metrics_typ = eval_on_typical_holdout(model)
    metrics_full = eval_on_full_holdout(model)
    n_params = int(sum(p.numel() for p in model.parameters()))
    return {
        "kind": "torch",
        "fitted": model,
        "val_mae": float(val_mae),
        "epochs_used": int(epochs_used),
        "n_params": n_params,
        "metrics_typical": metrics_typ,
        "metrics_full": metrics_full,
    }

baseline_lookup = {}
if baseline_df is not None and len(baseline_df) > 0:
    baseline_lookup = baseline_df.set_index("model").to_dict(orient="index")

dl_studies = {}
tuned_rows = []

for arch_name in selected_archs:
    if arch_name not in OBJECTIVE_MAP:
        print(f"Skipping {arch_name}: no objective defined")
        continue

    n_trials = ARCH_TRIALS[arch_name]
    print(f"\n{'-' * 80}")
    print(f"  OPTUNA | {arch_name} | trials={n_trials}")
    print(f"{'-' * 80}")

    t0 = time.time()
    seed_everything(SEED)
    sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True, warn_independent_sampling=False)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=min(8, max(3, n_trials // 3)), n_warmup_steps=20)
    study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)

    if ENQUEUE_BASELINE_TRIAL and arch_name in BASELINE_TRIAL_PARAMS:
        try:
            study.enqueue_trial(serialize_params(BASELINE_TRIAL_PARAMS[arch_name]))
            print("  Enqueued baseline configuration as a valid trial.")
        except Exception as e:
            print(f"  Could not enqueue baseline trial: {e}")

    study.optimize(OBJECTIVE_MAP[arch_name], n_trials=n_trials, show_progress_bar=True)
    dl_studies[arch_name] = study

    search_best_params = serialize_params(study.best_params)
    search_elapsed = time.time() - t0
    search_fit = fit_best_inner_model(arch_name, search_best_params)

    baseline_ref = baseline_lookup.get(arch_name)
    selected_source = "tuned_search"
    selected_params = search_best_params
    selected_val_mae = float(study.best_value)
    selected_refit_val_mae = float(search_fit["val_mae"])
    selected_epochs = int(search_fit["epochs_used"])
    selected_n_params = int(search_fit["n_params"])
    selected_metrics_typ = search_fit["metrics_typical"]
    selected_metrics_full = search_fit["metrics_full"]

    if KEEP_BASELINE_IF_NOT_IMPROVED and baseline_ref is not None and arch_name in BASELINE_TRIAL_PARAMS:
        baseline_val_ref = float(baseline_ref["val_MAE"])
        if baseline_val_ref <= float(study.best_value) + BASELINE_KEEP_EPS:
            selected_source = "baseline_keep"
            selected_params = serialize_params(BASELINE_TRIAL_PARAMS[arch_name])
            selected_val_mae = baseline_val_ref
            selected_refit_val_mae = baseline_val_ref
            selected_epochs = int(baseline_ref.get("epochs", 0))
            selected_n_params = int(baseline_ref.get("params", 0))
            selected_metrics_typ = {
                "MAE": float(baseline_ref["MAE_typical"]),
                "RMSE": float(baseline_ref["RMSE_typical"]),
                "MdAE": float(baseline_ref["MdAE_typical"]),
            }
            selected_metrics_full = {
                "MAE": float(baseline_ref["MAE_full"]),
                "RMSE": float(baseline_ref["RMSE_full"]),
                "MdAE": float(baseline_ref["MdAE_full"]),
            }

    print(f"  Search best val MAE: {study.best_value:.4f}")
    print(f"  Search refit val MAE: {search_fit['val_mae']:.4f}")
    if baseline_ref is not None:
        print(f"  Baseline reference val MAE: {float(baseline_ref['val_MAE']):.4f}")
    print(f"  Selected source: {selected_source}")
    print(f"  Selected val MAE: {selected_val_mae:.4f}")
    print(f"  Typical holdout: {selected_metrics_typ}")
    print(f"  Full holdout:    {selected_metrics_full}")
    print(f"  Время Optuna:    {search_elapsed:.1f}s")

    trials_df = study.trials_dataframe()
    trials_df.to_csv(RUN_DIR / f"trials_{arch_name}.csv", index=False)
    trials_df.to_csv(ART_DIR / f"trials_{arch_name}.csv", index=False)

    with open(RUN_DIR / f"search_best_params_{arch_name}.json", "w", encoding="utf-8") as f:
        json.dump(search_best_params, f, ensure_ascii=False, indent=2)
    with open(ART_DIR / f"search_best_params_{arch_name}.json", "w", encoding="utf-8") as f:
        json.dump(search_best_params, f, ensure_ascii=False, indent=2)

    with open(RUN_DIR / f"best_params_{arch_name}.json", "w", encoding="utf-8") as f:
        json.dump(selected_params, f, ensure_ascii=False, indent=2)
    with open(ART_DIR / f"best_params_{arch_name}.json", "w", encoding="utf-8") as f:
        json.dump(selected_params, f, ensure_ascii=False, indent=2)

    tuned_rows.append({
        "model": arch_name,
        "selected_source": selected_source,
        "n_trials": int(n_trials),
        "params": int(selected_n_params),
        "epochs": int(selected_epochs),
        "baseline_val_MAE": round(float(baseline_ref["val_MAE"]), 4) if baseline_ref is not None else np.nan,
        "search_best_val_MAE": round(float(study.best_value), 4),
        "search_refit_val_MAE": round(float(search_fit["val_mae"]), 4),
        "selected_val_MAE": round(float(selected_val_mae), 4),
        "refit_val_MAE": round(float(selected_refit_val_mae), 4),
        "MAE_typical": round(float(selected_metrics_typ["MAE"]), 4),
        "RMSE_typical": round(float(selected_metrics_typ["RMSE"]), 4),
        "MdAE_typical": round(float(selected_metrics_typ["MdAE"]), 4),
        "MAE_full": round(float(selected_metrics_full["MAE"]), 4),
        "RMSE_full": round(float(selected_metrics_full["RMSE"]), 4),
        "MdAE_full": round(float(selected_metrics_full["MdAE"]), 4),
        "delta_val_vs_baseline": round(float(baseline_ref["val_MAE"]) - float(selected_val_mae), 4) if baseline_ref is not None else np.nan,
        "delta_typical_vs_baseline": round(float(baseline_ref["MAE_typical"]) - float(selected_metrics_typ["MAE"]), 4) if baseline_ref is not None else np.nan,
        "delta_full_vs_baseline": round(float(baseline_ref["MAE_full"]) - float(selected_metrics_full["MAE"]), 4) if baseline_ref is not None else np.nan,
        "time_s": round(float(search_elapsed), 1),
        "search_best_params": search_best_params,
        "best_params": selected_params,
    })
    clear_device_cache()

tuned_df = pd.DataFrame(tuned_rows).sort_values(["selected_val_MAE", "MAE_typical", "MAE_full"]).reset_index(drop=True)
tuned_df.to_csv(RUN_DIR / "tuned_summary.csv", index=False)
tuned_df.to_csv(ART_DIR / "tuned_summary.csv", index=False)

print("\nTuned summary (sorted by selected_val_MAE):")
display(
    tuned_df.drop(columns=["best_params", "search_best_params"])
    if {"best_params", "search_best_params"}.issubset(tuned_df.columns)
    else tuned_df
)

if baseline_df is not None and len(tuned_df) > 0:
    compare_df = baseline_df[["model", "val_MAE", "MAE_typical", "MAE_full"]].rename(
        columns={
            "val_MAE": "baseline_val_MAE",
            "MAE_typical": "baseline_MAE_typical",
            "MAE_full": "baseline_MAE_full",
        }
    ).merge(
        tuned_df[[
            "model", "selected_source", "search_best_val_MAE", "selected_val_MAE",
            "MAE_typical", "MAE_full",
            "delta_val_vs_baseline", "delta_typical_vs_baseline", "delta_full_vs_baseline"
        ]],
        on="model",
        how="right"
    )
    compare_df = compare_df.sort_values(["selected_val_MAE", "MAE_typical", "MAE_full"]).reset_index(drop=True)
    compare_df.to_csv(RUN_DIR / "baseline_vs_tuned.csv", index=False)
    compare_df.to_csv(ART_DIR / "baseline_vs_tuned.csv", index=False)
    print("\nBaseline vs tuned (honest baseline-aware selection):")
    display(compare_df)


In [ ]:
# Опциональный финальный retrain лучшей selected-модели на полном train_typical.
# Основной критерий выбора — selected_val_MAE:
# либо лучшая найденная tuned-конфигурация, либо baseline-конфигурация,
# если тюнинг не улучшил baseline на inner temporal validation.

RUN_FINAL_RETRAIN = True

if not RUN_FINAL_RETRAIN or tuned_df.empty:
    print("Final retrain skipped.")
else:
    best_row = tuned_df.iloc[0]
    best_arch = best_row["model"]
    raw_best_params = best_row["best_params"]
    selected_source = best_row.get("selected_source", "unknown")

    if isinstance(raw_best_params, dict):
        best_params = raw_best_params
    elif isinstance(raw_best_params, str):
        try:
            best_params = json.loads(raw_best_params)
        except Exception:
            best_params = ast.literal_eval(raw_best_params)
    else:
        best_params = dict(raw_best_params)
    best_params = serialize_params(best_params)

    print(f"Best selected architecture by selected_val_MAE: {best_arch}")
    print(f"Selected source: {selected_source}")
    print(best_params)

    if best_arch == "GrowNet":
        n_stages = int(best_params["n_stages"])
        total_ep = int(best_row["epochs"])
        epochs_per_stage = max(20, math.ceil(total_ep / max(n_stages, 1)))
        stages = train_grownet_fulltrain(
            n_stages=n_stages,
            hidden=best_params["hidden"],
            lr=best_params["lr"],
            wd=best_params["wd"],
            step_size=best_params["step_size"],
            bs=best_params["bs"],
            epochs_per_stage=epochs_per_stage,
            dropout=best_params["dropout"],
        )
        final_typ = eval_fulltrain_grownet_typical(stages, best_params["step_size"])
        final_full = eval_fulltrain_grownet_full(stages, best_params["step_size"])
        final_payload = {
            "model": best_arch,
            "selected_source": selected_source,
            "selection_metric": "selected_val_MAE",
            "retrain_mode": "full_train_typical",
            "epochs_used": int(epochs_per_stage * n_stages),
            "MAE_typical": final_typ["MAE"],
            "RMSE_typical": final_typ["RMSE"],
            "MdAE_typical": final_typ["MdAE"],
            "MAE_full": final_full["MAE"],
            "RMSE_full": final_full["RMSE"],
            "MdAE_full": final_full["MdAE"],
            "best_params": best_params,
        }
    elif best_arch == "TabPFN":
        assert HAS_TABPFN, "tabpfn не установлен"
        pfn_device = "cuda" if torch.cuda.is_available() else "cpu"
        tabpfn = TabPFNRegressor(device=pfn_device, n_estimators=best_params.get("n_estimators", 8))
        max_rows = best_params.get("max_rows", 3000)

        X_pfn_tr = X_full_s.copy()
        y_pfn_tr = ytrain.values.astype(np.float32)
        if len(X_pfn_tr) > max_rows:
            idx = np.random.RandomState(SEED).choice(len(X_pfn_tr), max_rows, replace=False)
            X_pfn_tr = X_pfn_tr[idx]
            y_pfn_tr = y_pfn_tr[idx]

        tabpfn.fit(X_pfn_tr, y_pfn_tr)
        final_typ, final_full = eval_tabpfn_holdouts(tabpfn_model=tabpfn, use_full_train_scaler=True)
        final_payload = {
            "model": best_arch,
            "selected_source": selected_source,
            "selection_metric": "selected_val_MAE",
            "retrain_mode": "full_train_typical",
            "epochs_used": 0,
            "MAE_typical": final_typ["MAE"],
            "RMSE_typical": final_typ["RMSE"],
            "MdAE_typical": final_typ["MdAE"],
            "MAE_full": final_full["MAE"],
            "RMSE_full": final_full["RMSE"],
            "MdAE_full": final_full["MdAE"],
            "best_params": best_params,
        }
    else:
        final_model, aux_w = build_arch_from_params(best_arch, best_params)
        full_epochs = max(1, int(best_row["epochs"]))
        final_model = train_model_fulltrain(
            final_model,
            epochs=full_epochs,
            lr=best_params["lr"],
            wd=best_params["wd"],
            bs=best_params["bs"],
            aux_w=aux_w,
        )
        if best_arch == "TabR":
            final_model.build_store(X_full_t.to(device), y_full_t.to(device))
        final_typ = eval_fulltrain_model_typical(final_model)
        final_full = eval_fulltrain_model_full(final_model)
        final_payload = {
            "model": best_arch,
            "selected_source": selected_source,
            "selection_metric": "selected_val_MAE",
            "retrain_mode": "full_train_typical",
            "epochs_used": full_epochs,
            "MAE_typical": final_typ["MAE"],
            "RMSE_typical": final_typ["RMSE"],
            "MdAE_typical": final_typ["MdAE"],
            "MAE_full": final_full["MAE"],
            "RMSE_full": final_full["RMSE"],
            "MdAE_full": final_full["MdAE"],
            "best_params": best_params,
        }

    with open(RUN_DIR / "best_model_fulltrain_metrics.json", "w", encoding="utf-8") as f:
        json.dump(final_payload, f, ensure_ascii=False, indent=2)
    with open(ART_DIR / "best_model_fulltrain_metrics.json", "w", encoding="utf-8") as f:
        json.dump(final_payload, f, ensure_ascii=False, indent=2)

    print("Final retrain metrics:")
    display(pd.DataFrame([final_payload]).drop(columns=["best_params"]))

    run_manifest = {
        "run_id": RUN_ID,
        "tabpfn_hf_login_status": globals().get("TABPFN_HF_LOGIN_STATUS", "unknown"),
        "run_tabpfn": bool(RUN_TABPFN),
        "artifacts_root": str(ART_DIR),
        "run_dir": str(RUN_DIR),
        "data_path": str(DATA_PATH),
        "baseline_dir": str(BASELINE_DIR) if BASELINE_DIR is not None else None,
        "selected_archs": selected_archs,
        "tune_mode": TUNE_MODE,
        "top_k": TOP_K,
        "trial_scale": TRIAL_SCALE,
        "min_trials": MIN_TRIALS,
        "enqueue_baseline_trial": bool(ENQUEUE_BASELINE_TRIAL),
        "keep_baseline_if_not_improved": bool(KEEP_BASELINE_IF_NOT_IMPROVED),
        "best_model": final_payload.get("model"),
        "best_model_selected_source": final_payload.get("selected_source"),
    }
    with open(RUN_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
        json.dump(run_manifest, f, ensure_ascii=False, indent=2)
    with open(ART_DIR / "run_manifest_latest.json", "w", encoding="utf-8") as f:
        json.dump(run_manifest, f, ensure_ascii=False, indent=2)

    print(f"Saved artifacts to: {RUN_DIR}")
